# LoRA (Low-Rank Adaptation) - Comprehensive Implementation Guide

This notebook covers:
- **Basic Implementation**: Simple, educational version
- **Advanced Implementation**: Production-ready patterns
- **Real-World Examples**: How companies use this in production
- **Integration**: Using popular libraries

Source: `llm/concepts/lora.md`

## Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("Libraries loaded successfully")

## 1. Basic Implementation

Simple, educational version to understand core concepts.

In [ ]:
# Basic LoRA Implementation
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """Linear layer with LoRA: W = W0 + A @ B^T"""

    def __init__(self, in_features, out_features, rank=8):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))

        # LoRA weights
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)

        # Initialize
        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        base = torch.nn.functional.linear(x, self.weight, self.bias)
        lora = self.lora_B(self.lora_A(x))
        return base + lora

# Test
layer = LoRALinear(768, 3072, rank=8)
x = torch.randn(2, 10, 768)
output = layer(x)
print(f"Output shape: {output.shape}")
print(f"Params: {sum(p.numel() for p in layer.parameters()):,}")

## 2. Advanced Implementation

Production-ready patterns with optimization and error handling.

In [ ]:
# Advanced LoRA with Scaling and Merging
import torch
import torch.nn as nn
import math

class ScaledLoRALinear(nn.Module):
    """LoRA with alpha scaling and merge capability"""

    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))

        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        self.alpha = alpha
        self.rank = rank
        self.scaling = alpha / rank

        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        base = torch.nn.functional.linear(x, self.weight, self.bias)
        lora = self.lora_B(self.lora_A(x)) * self.scaling
        return base + lora

    def merge(self):
        """Merge LoRA into base weight for inference"""
        delta_w = (self.lora_B.weight @ self.lora_A.weight) * self.scaling
        self.weight.data += delta_w.t()
        # Disable LoRA after merge
        self.lora_A.weight.requires_grad = False
        self.lora_B.weight.requires_grad = False

# Usage
layer = ScaledLoRALinear(768, 3072, rank=8, alpha=16)
x = torch.randn(2, 768)
output = layer(x)
print(f"Training mode output: {output.shape}")
layer.merge()
output_merged = layer(x)
print(f"Merged mode (no LoRA computation)")

## Real-World: Peft

How this is used in production systems.

In [ ]:
# Real-World: Using HuggingFace PEFT for LoRA
from peft import get_peft_model, LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch.optim import AdamW

# Load model
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # Which modules to apply LoRA to
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training
optimizer = AdamW(model.parameters(), lr=1e-4)
model.train()

# Forward pass
text = "Hello, how are you?"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs, labels=inputs.input_ids)
loss = outputs.loss

loss.backward()
optimizer.step()
optimizer.zero_grad()

print(f"Loss: {loss.item():.4f}")

## Real-World: Multilora

How this is used in production systems.

In [ ]:
# Real-World: Multi-Task LoRA
class MultiTaskLoRA:
    """Manage multiple LoRA adapters for different tasks"""

    def __init__(self, base_model, model_name):
        self.base_model = base_model
        self.loras = {}
        self.active_lora = None

    def add_task(self, task_name, lora_config):
        """Add a LoRA adapter for a new task"""
        from peft import get_peft_model
        model = get_peft_model(self.base_model, lora_config)
        self.loras[task_name] = model

    def switch_task(self, task_name):
        """Switch active task"""
        if task_name not in self.loras:
            raise ValueError(f"Task {task_name} not found")
        self.active_lora = task_name

    def forward(self, *args, **kwargs):
        """Forward pass with active LoRA"""
        if self.active_lora is None:
            return self.base_model(*args, **kwargs)
        return self.loras[self.active_lora](*args, **kwargs)

# Usage: same base model, different LoRA for sentiment, NER, QA
# Each task-specific LoRA is ~0.5% of base model size

## Resources & Next Steps

- **Detailed Explanation**: See `llm/concepts/lora.md`
- **Interview Questions**: Review Q&A in markdown file
- **Real-World Examples**: See companies section in markdown
- **Experiment**: Modify the code above and run cells

### Concepts to explore next:
- Related concepts (see markdown file)
- Cross-concept combinations
- Integration with other techniques